### Imports

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LinearRegression
from scipy.optimize import curve_fit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import spearmanr
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import find_peaks, medfilt
from scipy.interpolate import make_interp_spline
from sklearn.model_selection import train_test_split
from sklearn.model_selection import  KFold, GroupKFold

from model_functions import *
from helper_functions import *
from helper_plot_functions import *

In [ ]:
def updrs_trial5_summary(exclude_ids=None):
    CSV_FEATURES = r"PATH_TO_CSV.csv"
    CSV_METADATA = r"PATH_TO_METADATA.csv"

    features, amplitude, speed, trial_names, trial_numbers, handedness, motor_skills = load_speed_amp(
        CSV_FEATURES, CSV_METADATA, None, exclude_ids=exclude_ids
    )
    ids = features["ids"].values
    trial_numbers_arr = np.array(trial_numbers)

    mask_updrs = (trial_numbers_arr == "U1") | (trial_numbers_arr == "U2")
    mask_trial5 = (trial_numbers_arr == "5")
    mask_rest = ~(mask_updrs | mask_trial5)

    rows = []
    for pid in np.unique(ids):
        pid_mask = (ids == pid)

        amp_updrs = amplitude[pid_mask & mask_updrs]
        spd_updrs = speed[pid_mask & mask_updrs]
        amp_t5 = amplitude[pid_mask & mask_trial5]
        spd_t5 = speed[pid_mask & mask_trial5]
        amp_rest = amplitude[pid_mask & mask_rest]
        spd_rest = speed[pid_mask & mask_rest]

        if len(amp_updrs) == 0 or len(amp_t5) == 0 or len(amp_rest) < 3:
            continue

        model = fit_logarithmic_model(spd_rest, amp_rest)
        pred_updrs = predict_logarithmic_model(model, spd_updrs)
        residuals = amp_updrs - pred_updrs

        rows.append({
            "participant": f"P{int(pid):03d}",
            "mean_amp_updrs": np.mean(amp_updrs),
            "mean_spd_updrs": np.mean(spd_updrs),
            "mean_amp_trial5": np.mean(amp_t5),
            "mean_spd_trial5": np.mean(spd_t5),
            "amp_diff": np.mean(amp_updrs) - np.mean(amp_t5),
            "spd_diff": np.mean(spd_updrs) - np.mean(spd_t5),
            "mae_updrs": np.mean(np.abs(residuals)),
            "mean_residual_updrs": np.mean(residuals),
        })

    df = pd.DataFrame(rows)

    print("=== UPDRS vs Trial 5 Summary ===\n")
    print(df.to_string(index=False, float_format="{:.4f}".format))
    print(f"\n--- Aggregated ---")
    print(f"Mean amplitude difference (UPDRS - Trial 5): {df['amp_diff'].mean():.4f} ± {df['amp_diff'].std():.4f}°")
    print(f"Mean speed difference (UPDRS - Trial 5): {df['spd_diff'].mean():.4f} ± {df['spd_diff'].std():.4f}s")
    print(f"UPDRS MAE vs fitted line: {df['mae_updrs'].mean():.4f} ± {df['mae_updrs'].std():.4f}°")
    print(f"UPDRS mean residual vs fitted line: {df['mean_residual_updrs'].mean():.4f} ± {df['mean_residual_updrs'].std():.4f}°")
    direction = "above" if df['mean_residual_updrs'].mean() > 0 else "below"
    print(f"=> UPDRS trials tend to fall {direction} the fitted tradeoff curve")

    return df

In [ ]:
updrs_trial5_summary()

In [ ]:
plot_updrs(2)

In [ ]:
plot_updrs(3)

In [ ]:
plot_updrs(4)

In [ ]:
plot_updrs(5)

In [ ]:
plot_updrs(7)

In [ ]:
plot_updrs(8)

In [ ]:
plot_updrs(9)

In [ ]:
plot_updrs(11)

In [ ]:
plot_updrs(12)

In [ ]:
plot_updrs(13)

In [ ]:
plot_updrs(14) 

In [ ]:
plot_updrs(15)

In [ ]:
plot_updrs(16) 